In [ ]:
!hostname

In [ ]:
import pandas
print(pandas.__file__)

In [ ]:
# ---BASIC IMPORTS---
import io
import os
import sys
import contextlib
from IPython.display import display, Javascript

In [ ]:
from datetime import date

#----EXPERIMENT------
exp = "FuNOVA_Screen"
suffix= "" # "_new_cy3" # ""
batch = 2
batches = [f'batch{batch}{suffix}']
plate = 4
config_name = f"qc_config_FuNOVA_Screen_b{batch}{suffix}"


# ---WORKING ENV SETUP---
NOVA_HOME = '/home/projects/hornsteinlab/giliwo/NOVA'
# home/projects/hornsteinlab/giliwo/NOVA
#'/home/projects/hornsteinlab/Collaboration/NOVA' 

os.environ['NOVA_HOME'] = NOVA_HOME
sys.path.insert(1, os.getenv("NOVA_HOME"))
print(f"NOVA_HOME: {os.getenv('NOVA_HOME')}")

#---IO PATHS---
today = date.today()

INPUT_NOTEBOOK = os.path.join(NOVA_HOME, "tools", "preprocessing_tools", "qc_reports", f"qc_report_{exp}.ipynb")

LOGS_PATH = os.path.join(NOVA_HOME, "outputs", "preprocessing", exp, "logs")
OUTPUT_HTML_DIR = os.path.join(NOVA_HOME, "outputs", "preprocessing", exp, f"batch{batch}", "QC_figures", f"plate{plate}")
batches_suffix = "_".join(batch for batch in batches)
OUTPUT_HTML_FILENAME = f"qc_report_{exp}_batch{batch}{suffix}_plate{plate}_{today}.html"

SAVE_HTML = True

# -----------------
# Init Preprocessing Config
# -----------------

from src.common.utils import load_config_file, get_class
from manuscript.preprocessing_config_FuNOVA_Screen import PreprocessingBaseConfigFuNOVAScreen

from manuscript.FuNOVA_Screen_Conditions_Lists_b1 import (
    plate1_conditions as b1_plate1_conditions,
    plate2_conditions as b1_plate2_conditions,
    plate3_conditions as b1_plate3_conditions,
    plate4_conditions as b1_plate4_conditions,
)

from manuscript.FuNOVA_Screen_Conditions_Lists_b2 import (
    plate1_conditions as b2_plate1_conditions,
    plate2_conditions as b2_plate2_conditions,
    plate3_conditions as b2_plate3_conditions,
    plate4_conditions as b2_plate4_conditions,
)

run_config = PreprocessingBaseConfigFuNOVAScreen()
conditions = globals()[f"b{batch}_plate{plate}_conditions"]
run_config.CONDITIONS = conditions
root_directory_raw = run_config.RAW_FOLDER_ROOT 
root_directory_proc = run_config.PROCESSED_FOLDER_ROOT 

print("raw input: ", root_directory_raw)
print("processed input: ", root_directory_proc)
print("config input: ", "PreprocessingBaseConfigFuNOVAScreen")
print("log dir path: ", LOGS_PATH)
print("html output: ", "save = ", SAVE_HTML, ", file_name = ", OUTPUT_HTML_FILENAME)


In [ ]:
#---NOVA FUNCTIONS IMPORTS---
import importlib   
from tools.preprocessing_tools.qc_reports.qc_utils import log_files_qc, run_validate_folder_structure, display_diff, sample_and_calc_variance, \
                                                show_site_survival_dapi_brenner, show_site_survival_dapi_cellpose, \
                                                show_site_survival_dapi_tiling, show_site_survival_target_brenner, \
                                                calc_total_sums, plot_filtering_heatmap, show_total_sum_tables, \
                                                plot_cell_count, plot_catplot, plot_hm_of_mean_cell_count_per_tile, \
                                                run_calc_hist_new, show_total_valid_tiles_per_marker_and_batch, \
                                                plot_marker_data
                                                
# from GAL
qc_config = importlib.import_module(
    f"tools.preprocessing_tools.qc_reports.{config_name}"
)

(markers, cell_lines, cell_lines_for_disp, reps, line_colors, lines_order, custom_palette, expected_dapi_raw, expected_marker_raw, panels, marker_info) = (qc_config.funova_markers, qc_config.funova_cell_lines, qc_config.funova_cell_lines_for_disp, qc_config.funova_reps, qc_config.funova_line_colors, qc_config.funova_lines_order, qc_config.funova_custom_palette, qc_config.funova_expected_dapi_raw, qc_config.funova_expected_marker_raw, qc_config.funova_panels, qc_config.funova_marker_info)
cell_lines_to_cond = {"C9": conditions}
DAPI_NAME = qc_config.DAPI_NAME


# from tools.preprocessing_tools.qc_reports.config_name import (
#     funova_markers as markers,
#     funova_cell_lines as cell_lines,
#     funova_cell_lines_to_cond as cell_lines_to_cond,
#     funova_cell_lines_for_disp as cell_lines_for_disp,
#     funova_reps as reps,
#     funova_line_colors as line_colors,
#     funova_lines_order as lines_order,
#     funova_custom_palette as custom_palette,
#     funova_expected_dapi_raw as expected_dapi_raw,
#     funova_expected_marker_raw as expected_marker_raw,
#     funova_panels as panels,
#     funova_marker_info as marker_info
# )
from tools.show_images_utils import *

In [ ]:
# --- restrict color/display maps to the CURRENT plate's conditions ---
# The config builds line_colors / cell_lines_for_disp across all 4 plates,
# which leaks other plates' conditions into the QC plots/tables. Filter to `conditions`.
_cur = set(conditions)
line_colors         = {k: v for k, v in line_colors.items()         if k.split(' ', 1)[-1] in _cur}  # keys "C9 <cond>"
cell_lines_for_disp = {k: v for k, v in cell_lines_for_disp.items() if k.split('_', 1)[-1] in _cur}  # keys "C9_<cond>"
# keep order/palette consistent with the filtered colors
lines_order    = list(line_colors.keys())
custom_palette = [line_colors[l] for l in lines_order]
print(f"conditions this plate: {len(_cur)} | line_colors: {len(line_colors)} | cell_lines_for_disp: {len(cell_lines_for_disp)}")

In [ ]:
print(expected_dapi_raw)

In [ ]:
#---NOTEBOOK SETUP---
%load_ext autoreload
%autoreload 2

In [ ]:
# NEED TO BE IN "batchX" folder, need to change batch files manually
df = log_files_qc(LOGS_PATH, only_wt_cond=False, batches=batches, filename_split='_',site_location=0)

In [ ]:
# filter df based on conditions
df = df[df['condition'].isin(conditions)]
print(f"df shape after filtering by conditions: {df.shape}")

In [ ]:
# Gal's feature re-naming
df[['cell_line', 'marker', 'cell_line_cond']] = df[['cell_line', 'marker', 'cell_line_cond']].apply(lambda x: x.str.replace('_', '-', regex=False))
df['filename'] = df['filename'].str.split('-').str[0]
df['site_num'] = df['site_num'].str.split('-').str[0]


In [ ]:
df["marker"].unique()

In [ ]:
df_dapi = df[df.marker==DAPI_NAME]
df_target = df[df.marker!=DAPI_NAME]

print("df_dapi length:", df_dapi.shape[0])
print("df_target length:", df_target.shape[0])

In [ ]:
df_dapi["condition"].unique()

### Raw Files Validation

1. How many site **tiff** files do we have in each folder?
2. Are all existing files valid? (tif, at least 2049kB, not corrupetd)

In [ ]:
df_metadata = extract_image_metadata(root_directory_raw, FILE_EXTENSION = '.tiff', KEY_BATCH = 'Batch')

In [ ]:
# rename batches if suffix in it
if suffix:
    batches = [batch.split('_')[0] for batch in batches]

In [ ]:
raws = run_validate_folder_structure(root_directory_raw, 
                                     False, 
                                     panels, 
                                     markers.copy(), 
                                     OUTPUT_HTML_DIR, 
                                     marker_info,
                                    cell_lines_to_cond, 
                                    reps, 
                                    cell_lines_for_disp, 
                                    expected_dapi_raw,
                                    batches=batches, 
                                    fig_width=20,fig_height = 20,
                                    expected_count=expected_marker_raw, 
                                    check_antibody=False,
                                    DAPI_NAME = DAPI_NAME.replace("-", "_"),
                                    font_size=30,
                                    header_rotation=90,
                                    rep_col_width=0.15
                                    )

### Processed Files Validation

1. How many site **npy** files do we have in each folder? -> How many sites survived the pre-processing?
2. Are all existing files valid? (at least 100kB, npy not corrupted)

In [ ]:
procs = run_validate_folder_structure(root_directory_proc, True, 
                                      panels, 
                                      markers, 
                                      OUTPUT_HTML_DIR,
                                      marker_info,
                                    cell_lines_to_cond, 
                                    reps, 
                                    cell_lines_for_disp, 
                                    expected_dapi_raw,
                                     batches=batches, 
                                     fig_width=20,
                                     fig_height = 20,
                                    expected_count=expected_marker_raw, 
                                    check_antibody=False,
                                    DAPI_NAME = DAPI_NAME.replace("-", "_"),
                                    font_size=30,
                                    header_rotation=90,
                                    rep_col_width=0.15
                                    )

### Difference between Raw and Processed

In [ ]:
display_diff(batches, raws, procs, OUTPUT_HTML_DIR, fig_width=20, fig_height=20, font_size=30, header_rotation=90, rep_col_width=0.15)

### Variance in each batch (of processed files)

In [ ]:
# for batch in batches[:1]:
#     with contextlib.redirect_stdout(io.StringIO()):
#         var = sample_and_calc_variance(root_directory_proc, 
#                                        batch, 
#                                        sample_size_per_markers=50, 
#                                        cond_count=2, 
#                                        rep_count=len(reps), 
#                                        num_markers=len(markers))
#     print(f'{batch} var: ',var)


## Preprocessing Filtering qc
By order of filtering

### 1. % site survival after Brenner on DAPI channel
Percentage out of the total sites

In [ ]:
dapi_filter_by_brenner = show_site_survival_dapi_brenner(df_dapi,
                                                         batches, 
                                                         line_colors, 
                                                         panels,
                                                        figsize=(20,10),
                                                          reps=reps, vmax=expected_marker_raw)

In [ ]:
print(df_dapi.head())

### 2. % Site survival after Cellpose
Percentage out of the sites that passed the CellPose filter - minimum cells in a tile. In parenthesis are absolute values.

**A site will be filtered out if Cellpose found 0 cells in it.**

In [ ]:
dapi_filter_by_cellpose = show_site_survival_dapi_cellpose(df_dapi, 
                                                           batches, dapi_filter_by_brenner,
                                                           line_colors, 
                                                           panels, 
                                                           figsize=(30,15), reps=reps,
                                                           small_font_size = 10, big_font_size = 12)

### 3. % Site survival by tiling
Percentage out of the sites that passed the tiling flilter - variance / intensity / dead cells. In parenthesis are absolute values.

**A site will be filtered out if after tiling, no tile is containing at least one whole cell that Cellpose detected.**

In [ ]:
print(df_dapi["cell_line_cond"].unique())

In [ ]:
print(dapi_filter_by_cellpose["cell_line_cond"].unique)

In [ ]:
# dapi_filter_by_tiling=show_site_survival_dapi_tiling(df_dapi, 
                                                    #  [batch.split('_')[0] for batch in batches], 
                                                    #  dapi_filter_by_cellpose, 
                                                    #  line_colors, 
                                                    #  panels, 
                                                    #  reps, 
                                                    #  figsize=(40,30),
                                                    #  small_font_size = 10, big_font_size = 12)



## Statistics About the Processed Files

In [ ]:
# DAPI_NAME =DAPI_NAME.replace("-", "_")
# markers = [marker.replace("-", "_") for marker in markers]
markers = [marker.replace("_", "-") for marker in markers]
print(DAPI_NAME)
print(markers)

In [ ]:
names = ['Total number of tiles', 'Total number of whole cells']
stats = ['n_valid_tiles','site_whole_cells_counts_sum','site_cell_count','site_cell_count_sum']
total_sum = calc_total_sums(df_target, df_dapi, stats, markers, DAPI_NAME = DAPI_NAME)

In [ ]:
print(df_target['marker'].unique())

In [ ]:
print(total_sum['marker'].unique())

### Total tiles

In [ ]:
total_sum[total_sum.marker.isin(markers)].n_valid_tiles.sum()

### Total whole nuclei in tiles

In [ ]:
total_sum[total_sum.marker == DAPI_NAME].site_whole_cells_counts_sum.sum()

### Total nuclei in sites

In [ ]:
total_sum[total_sum.marker == DAPI_NAME].site_cell_count.sum()

In [ ]:
total_sum['marker'].unique()

In [ ]:
show_total_sum_tables(total_sum)

### plot total sum

In [ ]:
# plot_marker_data(total_sum, split_by_cell_line=True)

### Show Total Tile Counts
For each batch, cell line, replicate and marker: Total number of tiles

#### First, we look at all cell lines togther:

In [ ]:
show_total_valid_tiles_per_marker_and_batch(total_sum, vmax=15000, DAPI_NAME = DAPI_NAME)

#### Separating into cell lines & batches:

In [ ]:
to_heatmap = total_sum.rename(columns={'n_valid_tiles':'index'})
plot_filtering_heatmap(to_heatmap, 
                       extra_index='marker', 
                       vmin=None, vmax=None,
                       xlabel = 'Total number of tiles', 
                       show_sum=True, figsize=(50,15), 
                       small_font_size=12, big_font_size=16,
                       fmt=".0f")

### Show Total Whole Cell Counts
For each batch, cell line, replicate and markerTotal number of tiles

In [ ]:
to_heatmap = total_sum.rename(columns={'site_whole_cells_counts_sum':'index'})
plot_filtering_heatmap(to_heatmap, 
                       extra_index='marker', 
                       vmin=None, vmax=None,
                       xlabel = 'Total number of whole cells', 
                       show_sum=True, 
                       figsize=(100,50), 
                       fmt=".0f",
                        small_font_size = 24, big_font_size = 30)

### Show **Cell Count** Statistics per Batch

In [ ]:
df_no_empty_sites = df_dapi[df_dapi.n_valid_tiles !=0]

plot_cell_count(df_no_empty_sites, 
                lines_order, 
                custom_palette, 
                y='site_cell_count_sum', 
                title='Cell Count Average per Site (from tiles)', 
                figsize=(16,6))


plot_cell_count(df_no_empty_sites, 
                lines_order, 
                custom_palette, 
                y='site_whole_cells_counts_sum',
                title='Whole Cell Count Average per Site',
                figsize=(16,6))


plot_cell_count(df_no_empty_sites, 
                lines_order, 
                custom_palette, 
                y='site_cell_count',
                title='Cellpose Cell Count Average per Site',
                figsize=(16,6))


### Show **Tiles** per Site Statistics


** for all sites

In [ ]:
df_dapi.groupby(['cell_line_cond']).n_valid_tiles.mean()

** only for sites with n_valid_tiles > 0

In [ ]:
df_no_empty_sites = df_dapi[df_dapi.n_valid_tiles !=0]
print("exluding ", len(df_dapi[df_dapi.n_valid_tiles == 0]), "sites...")
df_no_empty_sites.groupby(['cell_line_cond']).n_valid_tiles.mean()

In [ ]:
df_dapi[['site_cell_count']].mean()


### Show Mean of cell count in valid tiles

In [ ]:
plot_hm_of_mean_cell_count_per_tile(df_dapi, 
                                    split_by='rep', 
                                    rows='cell_line_cond', 
                                    columns='panel', 
                                    figsize=(40,12),
                                    wspace=0.8)

### save notebook

In [ ]:
display(Javascript('IPython.notebook.save_checkpoint();'))

In [ ]:

if SAVE_HTML:
    display(Javascript('IPython.notebook.save_checkpoint();'))  # no-op outside classic Notebook — save manually (Ctrl+S) first
    rc = os.system(
        f'jupyter nbconvert --to html "{INPUT_NOTEBOOK}" --output "{OUTPUT_HTML_FILENAME}" --output-dir "{OUTPUT_HTML_DIR}"'
    )
    print('Done.' if rc == 0 else f'nbconvert FAILED (exit {rc})')